[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module2/04-type-hints.ipynb)

# Type Hints
**Module 2 — Intermediate Python | Estimated time: 20 minutes**

## Learning Objectives
- Write **variable and function annotations** using built-in types
- Use the **`typing` module**: `Optional`, `Union`, `List`, `Dict`, `Tuple`, `Any`, `cast`
- Define **generic functions** with `TypeVar`
- Describe structural contracts with **`Protocol`**
- Use `TypedDict`, `Literal`, and `Final` for precise types
- Combine type hints with **`@dataclass`**
- Run **`mypy`** static type checking inside a notebook

In [ ]:
!pip install mypy --quiet
print('mypy installed.')

## 1. Basic Variable and Function Annotations

Type hints are **not enforced at runtime** — they are metadata for type checkers, IDEs, and documentation.  
Python stores them in `__annotations__` but ignores them during execution.

In [ ]:
# Variable annotations
name: str = 'Alice'
age: int = 30
pi: float = 3.14159
active: bool = True
nicknames: list[str] = ['Al', 'Allie']

# Function annotations
def greet(first: str, last: str, formal: bool = False) -> str:
    """Return a greeting string."""
    full = f'{first} {last}'
    return f'Good day, {full}.' if formal else f'Hey, {first}!'

print(greet('Alice', 'Smith'))
print(greet('Alice', 'Smith', formal=True))

# Annotations are stored in __annotations__
print('\ngreet annotations:', greet.__annotations__)

# Python does NOT enforce them — this runs without error
def add(a: int, b: int) -> int:
    return a + b

print(add('hello ', 'world'))   # str + str — Python doesn't care

## 2. `Optional`, `Union`, `Any` from `typing`

Python 3.10+ allows `X | Y` union syntax. For broader compatibility the `typing` module provides named aliases.

In [ ]:
from typing import Optional, Union, Any, cast

# Optional[T] is shorthand for Union[T, None]
def find_user(user_id: int) -> Optional[str]:
    """Return username or None if not found."""
    db = {1: 'Alice', 2: 'Bob'}
    return db.get(user_id)      # returns str or None

print(find_user(1))    # Alice
print(find_user(99))   # None


# Union — accept multiple types
def stringify(value: Union[int, float, bool]) -> str:
    return str(value)

print(stringify(42), stringify(3.14), stringify(True))

# Python 3.10+ shorthand (works here since Colab uses 3.10+)
def stringify2(value: int | float | bool) -> str:
    return str(value)


# Any — opt out of type checking for a value
def process(data: Any) -> Any:
    return data


# cast — tell the type checker to treat a value as a specific type
# (no runtime effect — pure annotation)
raw: Any = '42'
as_int: int = cast(int, int(raw))
print(type(as_int), as_int)

## 3. Container Types — `List`, `Dict`, `Tuple`, `Set`

From Python 3.9 you can use the built-in `list[str]`, `dict[str, int]` etc. directly. The `typing` versions (`List`, `Dict`) still work for compatibility.

In [ ]:
from typing import List, Dict, Tuple, Set, Sequence, Mapping

# Modern syntax (Python 3.9+)
def average(scores: list[float]) -> float:
    return sum(scores) / len(scores)

print(average([85.0, 92.0, 78.5, 96.0]))


# Nested types
def word_count(text: str) -> dict[str, int]:
    counts: dict[str, int] = {}
    for word in text.lower().split():
        counts[word] = counts.get(word, 0) + 1
    return counts

print(word_count('to be or not to be that is the question'))


# Tuple — fixed-length with specific types per position
Point = tuple[float, float]              # type alias
Color = tuple[int, int, int]             # RGB

def distance(p1: Point, p2: Point) -> float:
    return ((p1[0]-p2[0])**2 + (p1[1]-p2[1])**2) ** 0.5

print(distance((0.0, 0.0), (3.0, 4.0)))


# Sequence / Mapping — read-only, more flexible
def first(items: Sequence[int]) -> int:
    return items[0]

print(first([10, 20, 30]))
print(first((10, 20, 30)))

## 4. `TypeVar` — Generic Functions

`TypeVar` lets you write functions that work on *any* type while still maintaining the relationship between input and output types.

In [ ]:
from typing import TypeVar, Sequence

T = TypeVar('T')           # unconstrained — any type
Num = TypeVar('Num', int, float)   # constrained — only int or float


def first(items: Sequence[T]) -> T:
    """Return first element; return type matches element type."""
    if not items:
        raise ValueError('Empty sequence')
    return items[0]

# Type checker knows: first(list[str]) -> str
print(first(['apple', 'banana', 'cherry']))   # str
print(first([1, 2, 3]))                       # int
print(first((3.14, 2.71)))                    # float


def clamp(value: Num, lo: Num, hi: Num) -> Num:
    """Clamp value between lo and hi."""
    return max(lo, min(value, hi))

print(clamp(15, 0, 10))       # 10 (int)
print(clamp(0.5, 0.0, 1.0))  # 0.5 (float)


# Bounded TypeVar
from typing import TypeVar

Comparable = TypeVar('Comparable', bound='SupportsLessThan')

def min_of_three(a: T, b: T, c: T) -> T:
    """Return the minimum of three comparable values."""
    if a <= b and a <= c:
        return a
    elif b <= c:
        return b
    return c

print(min_of_three(5, 3, 8))
print(min_of_three('banana', 'apple', 'cherry'))

## 5. `Protocol` — Structural Subtyping (Duck Typing)

`Protocol` lets you define *structural* contracts — if an object has the right methods, it satisfies the protocol, regardless of inheritance.

In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Drawable(Protocol):
    """Anything that has a draw() method is Drawable."""
    def draw(self) -> str: ...


class Circle:
    def __init__(self, radius: float):
        self.radius = radius
    def draw(self) -> str:
        return f'Circle(r={self.radius})'

class Square:
    def __init__(self, side: float):
        self.side = side
    def draw(self) -> str:
        return f'Square(s={self.side})'

class Point:
    # No draw() method — does NOT satisfy Drawable
    def __init__(self, x, y):
        self.x, self.y = x, y


def render_all(shapes: list[Drawable]) -> None:
    for shape in shapes:
        print(' ', shape.draw())


shapes = [Circle(5.0), Square(3.0), Circle(1.5)]
render_all(shapes)

# isinstance works because of @runtime_checkable
print('Circle is Drawable:', isinstance(Circle(1), Drawable))
print('Point is Drawable: ', isinstance(Point(0, 0), Drawable))

## 6. `TypedDict`, `Literal`, and `Final`

- **`TypedDict`** — dict with a fixed set of typed keys (like a lightweight struct)
- **`Literal`** — restrict a value to specific literal values
- **`Final`** — mark a name as a constant (cannot be reassigned)

In [ ]:
from typing import TypedDict, Literal, Final

# TypedDict — typed dictionary schema
class UserRecord(TypedDict):
    id: int
    username: str
    email: str
    role: str

def display_user(user: UserRecord) -> str:
    return f'[{user["role"]}] {user["username"]} <{user["email"]}>'

alice: UserRecord = {'id': 1, 'username': 'alice', 'email': 'alice@example.com', 'role': 'admin'}
print(display_user(alice))


# Literal — only specific values are valid
Direction = Literal['north', 'south', 'east', 'west']
LogLevel = Literal['DEBUG', 'INFO', 'WARNING', 'ERROR', 'CRITICAL']

def move(direction: Direction, steps: int = 1) -> str:
    return f'Moving {direction} {steps} step(s)'

print(move('north', 3))
print(move('east'))


# Final — constant that should not be reassigned
MAX_RETRIES: Final[int] = 3
API_BASE_URL: Final[str] = 'https://api.example.com/v2'
PI: Final = 3.141592653589793

print(f'Max retries: {MAX_RETRIES}')
print(f'API URL: {API_BASE_URL}')

# Attempting MAX_RETRIES = 5 would be flagged by mypy (not runtime error)

## 7. `@dataclass` with Type Annotations

The `@dataclass` decorator auto-generates `__init__`, `__repr__`, and `__eq__` from your type annotations — a natural fit with type hints.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Product:
    id: int
    name: str
    price: float
    tags: list[str] = field(default_factory=list)
    discount: Optional[float] = None

    @property
    def final_price(self) -> float:
        if self.discount:
            return self.price * (1 - self.discount)
        return self.price

    def __str__(self) -> str:
        tag_str = ', '.join(self.tags) or 'none'
        return f'{self.name} (${self.final_price:.2f}) [tags: {tag_str}]'


widget = Product(id=1, name='Widget Pro', price=49.99, tags=['hardware', 'sale'], discount=0.15)
print(widget)
print(repr(widget))
print('Price equality:', Product(1, 'A', 10.0) == Product(1, 'A', 10.0))
print('Price inequality:', Product(1, 'A', 10.0) == Product(2, 'B', 20.0))

## 8. Running `mypy` in a Notebook

Use `%%writefile` to save a Python file and `!mypy` to check it.

In [ ]:
%%writefile /tmp/typed_example.py
from typing import Optional

def divide(a: float, b: float) -> Optional[float]:
    if b == 0:
        return None
    return a / b

# Type error: passing str instead of float
result: float = divide(10, 'two')  # type: ignore[arg-type]  <- suppresses in real code

# Another error: not handling Optional return
x: float = divide(10, 2)  # mypy: divide can return None, but we assigned to float

In [ ]:
# Run mypy — it will report type errors without running the code
!mypy /tmp/typed_example.py --strict 2>&1 | head -30

In [ ]:
%%writefile /tmp/typed_clean.py
from typing import Optional

def divide(a: float, b: float) -> Optional[float]:
    if b == 0:
        return None
    return a / b

# Correctly handle the Optional return
result = divide(10.0, 2.0)
if result is not None:
    print(f'Result: {result}')

In [ ]:
!mypy /tmp/typed_clean.py
print('---')
!python /tmp/typed_clean.py

## Practice Exercises

**Exercise 1 — Type-Safe Stack**  
Write a generic `Stack[T]` class using `TypeVar` and `@dataclass`. It should have `push(item: T)`, `pop() -> T`, `peek() -> T`, and `is_empty() -> bool` methods. Add proper type annotations throughout. Verify with `mypy`.

**Exercise 2 — `TypedDict` Schema**  
Define a `TypedDict` for a `GitCommit` with fields: `sha` (str), `message` (str), `author` (str), `timestamp` (float), `parents` (list of str). Write a function `format_commit(commit: GitCommit) -> str` that returns a single-line summary. Include a `total=False` variant `PartialCommit` where all fields are optional.

**Exercise 3 — `Protocol` for Serialisation**  
Define a `Serialisable` protocol with methods `to_dict() -> dict[str, Any]` and a class method `from_dict(data: dict[str, Any]) -> 'Serialisable'`. Implement it for both a `User` dataclass and a `Session` plain class, then write a function `save_all(items: list[Serialisable]) -> list[dict]` that calls `to_dict()` on each.